# Qualité des données d’accidents routiers

On exploite les fichiers BAAC 2023 et 2024 disponibles dans `dataset/`. Le but est de vérifier les données, calculer un indicateur mensuel et expliquer si le résultat peut être publié.


## 0. Chargement


In [1]:
from pathlib import Path
import pandas as pd

BASE = Path("dataset")
ANNEES = [2023, 2024]
FICHIERS = ["caract", "lieux", "vehicules", "usagers"]

def charger(annee, table):
    chemin = BASE / str(annee) / f"{table}-{annee}.csv"
    return pd.read_csv(chemin, sep=";", dtype="string", encoding="utf-8")

donnees = {
    (annee, table): charger(annee, table)
    for annee in ANNEES
    for table in FICHIERS
}

for (annee, table), df in donnees.items():
    print(f"{annee} / {table:10s} : {len(df):>7,} lignes, {df.shape[1]} colonnes")


2023 / caract     :  54,822 lignes, 15 colonnes
2023 / lieux      :  70,860 lignes, 18 colonnes
2023 / vehicules  :  93,585 lignes, 11 colonnes
2023 / usagers    : 125,789 lignes, 16 colonnes
2024 / caract     :  54,402 lignes, 15 colonnes
2024 / lieux      :  70,248 lignes, 18 colonnes
2024 / vehicules  :  92,678 lignes, 11 colonnes
2024 / usagers    : 125,187 lignes, 16 colonnes


## 1. Diagnostic

`caract` contient les accidents et `usagers` leur gravité. Comme un accident peut concerner plusieurs personnes, je regroupe d’abord les usagers par accident. Les valeurs `-1`, `N/A` et vides sont traitées comme non renseignées.


In [2]:
SENTINELLES = {"", "-1", "N/A", "NA", "null", "None"}

def profil(df):
    lignes = []
    for col in df.columns:
        valeurs = df[col].fillna("").str.strip()
        manquants = valeurs.isin(SENTINELLES) | valeurs.isna()
        lignes.append({
            "colonne": col,
            "manquants_ou_codes": int(manquants.sum()),
            "part_%": round(100 * manquants.mean(), 2),
            "valeurs_distinctes": int(valeurs.nunique()),
        })
    return pd.DataFrame(lignes).sort_values("part_%", ascending=False)

for annee in ANNEES:
    print(f"\n--- {annee} / caractéristiques ---")
    display(profil(donnees[(annee, "caract")]).head(10))
    print(f"--- {annee} / usagers ---")
    display(profil(donnees[(annee, "usagers")]).head(10))



--- 2023 / caractéristiques ---


,colonne,manquants_ou_codes,part_%,valeurs_distinctes
12,adr,1389,2.53,28995
11,col,13,0.02,8
9,int,6,0.01,10
10,atm,4,0.01,10
0,Num_Acc,0,0.00,54822
1,jour,0,0.00,31
2,mois,0,0.00,12
3,an,0,0.00,1
4,hrmn,0,0.00,1409
5,lum,2,0.00,6


--- 2023 / usagers ---


,colonne,manquants_ou_codes,part_%,valeurs_distinctes
12,secu3,123392,98.09,11
15,etatp,115690,91.97,4
13,locp,63330,50.35,11
14,actp,63253,50.29,13
11,secu2,55419,44.06,11
8,an_nais,2598,2.07,108
9,trajet,2499,1.99,8
7,sexe,2431,1.93,3
10,secu1,2292,1.82,11
6,grav,118,0.09,5



--- 2024 / caractéristiques ---


,colonne,manquants_ou_codes,part_%,valeurs_distinctes
12,adr,2310,4.25,28785
11,col,6,0.01,8
0,Num_Acc,0,0.00,54402
1,jour,0,0.00,31
2,mois,0,0.00,12
3,an,0,0.00,1
4,hrmn,0,0.00,1414
5,lum,0,0.00,5
6,dep,0,0.00,107
7,com,0,0.00,11285


--- 2024 / usagers ---


,colonne,manquants_ou_codes,part_%,valeurs_distinctes
15,etatp,114932,91.81,4
12,secu3,113133,90.37,11
13,locp,61771,49.34,11
14,actp,61726,49.31,13
11,secu2,53813,42.99,11
9,trajet,2626,2.10,8
8,an_nais,2579,2.06,106
7,sexe,2395,1.91,3
10,secu1,2103,1.68,11
0,Num_Acc,0,0.00,54402


In [3]:
for annee in ANNEES:
    car = donnees[(annee, "caract")]
    accidents = set(car["Num_Acc"].dropna())
    print(f"\n{annee} :")
    print("Num_Acc dupliqués dans caract :", int(car["Num_Acc"].duplicated().sum()))
    for table in ["lieux", "vehicules", "usagers"]:
        df = donnees[(annee, table)]
        cles = set(df["Num_Acc"].dropna())
        print(f"  {table:10s} | lignes={len(df):>7,} | accidents sans caract={len(cles-accidents):>4} | accidents sans détail={len(accidents-cles):>4}")



2023 :
Num_Acc dupliqués dans caract : 0
  lieux      | lignes= 70,860 | accidents sans caract=   0 | accidents sans détail=   0
  vehicules  | lignes= 93,585 | accidents sans caract=   0 | accidents sans détail=   0
  usagers    | lignes=125,789 | accidents sans caract=   0 | accidents sans détail=   0

2024 :
Num_Acc dupliqués dans caract : 0
  lieux      | lignes= 70,248 | accidents sans caract=   0 | accidents sans détail=   0
  vehicules  | lignes= 92,678 | accidents sans caract=   0 | accidents sans détail=   0
  usagers    | lignes=125,187 | accidents sans caract=   0 | accidents sans détail=   0


La clé `Num_Acc`, la date et la gravité sont les champs les plus importants pour le calcul. Les autres tables servent surtout à décrire le contexte de l’accident.


## 2. Nettoyage et anomalies

Je normalise les espaces autour des clés et vérifie les dates. Une gravité inconnue n’est pas remplacée par une valeur inventée : l’accident est signalé et écarté du calcul si aucune gravité valide n’est disponible.


In [4]:
def preparer_annee(annee):
    car_source = donnees[(annee, "caract")].copy()
    usa_source = donnees[(annee, "usagers")].copy()
    anomalies = []

    car_source["Num_Acc"] = car_source["Num_Acc"].str.strip()
    usa_source["Num_Acc"] = usa_source["Num_Acc"].str.strip()

    doublons = car_source[car_source["Num_Acc"].duplicated(keep=False)]
    for cle in doublons["Num_Acc"].dropna().unique():
        anomalies.append({"annee": annee, "regle": "clé accident dupliquée", "Num_Acc": cle, "action": "isolé", "statut": "à arbitrer"})

    dates = pd.to_datetime(
        car_source["an"].str.strip() + "-" + car_source["mois"].str.strip().str.zfill(2) + "-" + car_source["jour"].str.strip().str.zfill(2),
        errors="coerce",
    )
    dates_invalides = dates.isna()
    for cle in car_source.loc[dates_invalides, "Num_Acc"].dropna():
        anomalies.append({"annee": annee, "regle": "date invalide", "Num_Acc": cle, "action": "exclu du mensuel", "statut": "à corriger à la source"})

    usa_source["grav"] = usa_source["grav"].str.strip()
    grav_valide = usa_source["grav"].isin(["1", "2", "3", "4"])
    usa_source["grav_valide"] = grav_valide
    for cle in usa_source.loc[~grav_valide, "Num_Acc"].dropna().unique():
        anomalies.append({"annee": annee, "regle": "gravité absente ou hors domaine", "Num_Acc": cle, "action": "accident isolé du KPI", "statut": "à vérifier"})

    grav_num = pd.to_numeric(usa_source.loc[grav_valide, "grav"], errors="coerce")
    tmp = usa_source.loc[grav_valide, ["Num_Acc"]].copy()
    tmp["grav_num"] = grav_num
    grav_accident = tmp.groupby("Num_Acc")["grav_num"].min()
    incidents = pd.DataFrame({
        "Num_Acc": car_source["Num_Acc"],
        "date": dates,
    })
    incidents["mois"] = incidents["date"].dt.to_period("M").astype("string")
    incidents["grav_accident"] = incidents["Num_Acc"].map(grav_accident)
    incidents["eligible"] = incidents["date"].notna() & incidents["grav_accident"].notna()
    return incidents, pd.DataFrame(anomalies, columns=["annee", "regle", "Num_Acc", "action", "statut"])

incidents_par_an = {}
journal_par_an = {}
for annee in ANNEES:
    incidents_par_an[annee], journal_par_an[annee] = preparer_annee(annee)
    print(annee, "accidents chargés :", len(incidents_par_an[annee]), "| anomalies tracées :", len(journal_par_an[annee]))


2023 accidents chargés : 54822 | anomalies tracées : 116
2024 accidents chargés : 54402 | anomalies tracées : 0


Pour résumer un accident, je garde la gravité la plus sévère déclarée pour ses usagers. Le journal ci-dessous permet de retrouver les cas mis de côté.


In [5]:
journal_corrections = pd.concat(journal_par_an.values(), ignore_index=True)
display(journal_corrections.head(20))
journal_corrections.to_csv("journal_corrections.csv", index=False, encoding="utf-8")

,annee,regle,Num_Acc,action,statut
0,2023,gravité absente ou hors domaine,202300000421,accident isolé du KPI,à vérifier
1,2023,gravité absente ou hors domaine,202300001254,accident isolé du KPI,à vérifier
2,2023,gravité absente ou hors domaine,202300001462,accident isolé du KPI,à vérifier
3,2023,gravité absente ou hors domaine,202300002015,accident isolé du KPI,à vérifier
4,2023,gravité absente ou hors domaine,202300002418,accident isolé du KPI,à vérifier
5,2023,gravité absente ou hors domaine,202300002966,accident isolé du KPI,à vérifier
6,2023,gravité absente ou hors domaine,202300003477,accident isolé du KPI,à vérifier
7,2023,gravité absente ou hors domaine,202300003552,accident isolé du KPI,à vérifier
8,2023,gravité absente ou hors domaine,202300004224,accident isolé du KPI,à vérifier
9,2023,gravité absente ou hors domaine,202300004694,accident isolé du KPI,à vérifier


## 3. Contrôles avant publication

Avant de publier, il faut vérifier qu’il n’y a pas de clé accident dupliquée, que les dates sont valides et que les accidents comptés ont une gravité renseignée. Les anomalies moins importantes, comme des coordonnées absentes, restent à suivre sans bloquer cet indicateur.


## 4. Indicateur et décision

Je calcule le nombre d’accidents corporels par mois et, en complément, le nombre d’accidents avec au moins une personne tuée. La comparaison entre années reste descriptive : il faut vérifier que les extractions sont complètes avant de conclure.


In [6]:
toutes = pd.concat(incidents_par_an.values(), ignore_index=True)
mensuel = (
    toutes[toutes["eligible"]]
    .assign(annee=lambda d: d["date"].dt.year)
    .groupby(["annee", "mois"], as_index=False)
    .agg(
        accidents_comptes=("Num_Acc", "nunique"),
        accidents_avec_tue=("grav_accident", lambda s: int((s == 2).sum())),
    )
)
comparaison = mensuel.pivot(index="mois", columns="annee", values="accidents_comptes")
comparaison["ecart_2024_vs_2023"] = comparaison[2024] - comparaison[2023]
comparaison["ecart_%"] = (100 * comparaison["ecart_2024_vs_2023"] / comparaison[2023]).round(1)
display(mensuel.sort_values(["annee", "mois"]))
display(comparaison)


,annee,mois,accidents_comptes,accidents_avec_tue
0,2023,2023-01,4053,129
1,2023,2023-02,3682,116
2,2023,2023-03,3998,108
3,2023,2023-04,4162,134
4,2023,2023-05,4767,152
5,2023,2023-06,5452,157
6,2023,2023-07,4754,169
7,2023,2023-08,4121,146
8,2023,2023-09,5161,159
9,2023,2023-10,5389,157


annee,2023,2024,ecart_2024_vs_2023,ecart_%
mois,,,,
2023-01,4053.0,NaN,NaN,NaN
2023-02,3682.0,NaN,NaN,NaN
2023-03,3998.0,NaN,NaN,NaN
2023-04,4162.0,NaN,NaN,NaN
2023-05,4767.0,NaN,NaN,NaN
2023-06,5452.0,NaN,NaN,NaN
2023-07,4754.0,NaN,NaN,NaN
2023-08,4121.0,NaN,NaN,NaN
2023-09,5161.0,NaN,NaN,NaN


In [7]:
resume = []
for annee, df in incidents_par_an.items():
    source = donnees[(annee, "caract")]
    journal = journal_par_an[annee]
    resume.append({
        "annee": annee,
        "lignes_caract": len(source),
        "cles_uniques": source["Num_Acc"].nunique(),
        "dates_valides": int(df["date"].notna().sum()),
        "accidents_avec_gravite": int(df["grav_accident"].notna().sum()),
        "accidents_eligibles": int(df["eligible"].sum()),
        "anomalies_tracees": len(journal),
    })
resume = pd.DataFrame(resume)
display(resume)


,annee,lignes_caract,cles_uniques,dates_valides,accidents_avec_gravite,accidents_eligibles,anomalies_tracees
0,2023,54822,54822,54822,54822,54822,116
1,2024,54402,54402,54402,54402,54402,0


### Décision

**NO-GO provisoire** tant que la règle d’agrégation de la gravité et la complétude des extractions ne sont pas confirmées. Après validation, le résultat peut être publié avec le journal des anomalies.

Ces volumes ne mesurent pas à eux seuls le risque routier. Pour cela, il faudrait aussi connaître le trafic ou les kilomètres parcourus.
